# 04 — Evidence-attached facility ranking

Part C only. See `docs/04_ranking.md` for inspected branch snapshots, confirmed ANN fields,
the confirmed B output contract, decision rules, and mandatory Databricks checks.
The default run uses three entirely synthetic facilities. It never writes to data tables.
Keep `ranking.py` and `examples/` beside this notebook in the Databricks Git folder.


In [ ]:
import json
from pathlib import Path
from time import perf_counter
import importlib.util

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "examples/04_ranking_current_sample.json").exists():
    raise RuntimeError("Open this notebook in the repository Git folder with ranking.py and examples/")
ranking_path = REPO_ROOT / "ranking.py"
spec = importlib.util.spec_from_file_location("part_c_ranking_current", ranking_path)
ranking_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ranking_module)
if ranking_module.FACILITY_TABLE != "workspace.datathon.facilities_clean" or "procedure_list" not in ranking_module.SOURCE_FIELDS:
    raise RuntimeError(f"Outdated ranking.py at {ranking_path}. Update it from milly/part before running.")
Ranker = ranking_module.Ranker
configure = ranking_module.configure
load_from_spark = ranking_module.load_from_spark
recommend = ranking_module.recommend
print("Loaded ranking code:", ranking_path)


In [ ]:
sample = json.loads((REPO_ROOT / "examples/04_ranking_current_sample.json").read_text())
print(sample["notice"])
configure(Ranker(sample["facilities"], sample["evidence"]))
demo = recommend("hemodialysis", 25.5941, 85.1376, k=1)
assert demo["status"] == "no_nearby_strong_evidence"
assert demo["recommendations"][0]["unique_id"] == "sample-weak"
assert demo["nearest_strong_option"]["unique_id"] == "sample-strong"
print(json.dumps(demo, indent=2, allow_nan=False))


## Live Databricks mode

Run `04_prepare_inputs` once if the agreed input tables are absent. It uses pinned
team logic and the list-column compatibility bridge without overwriting tables.
Then enable RUN_LIVE below to validate real records. The demo above is synthetic.


In [ ]:
RUN_LIVE = False
live_loaded = False
if RUN_LIVE:
    started = perf_counter()
    live_ranker = load_from_spark(spark)
    configure(live_ranker)
    live_loaded = True
    print(f"Loaded snapshot in {perf_counter() - started:.2f}s")
    print("Available canonical services:", sorted(live_ranker.evidence))
else:
    print("Live checks NOT RUN. Synthetic demo is still configured.")


In [ ]:
if RUN_LIVE:
    if not live_loaded:
        raise RuntimeError("Live snapshot was not loaded; do not validate against demo records")
    queries = json.loads((REPO_ROOT / "examples/04_ranking_queries.json").read_text())
    live_report = []
    for query in queries:
        started = perf_counter()
        result = recommend(query["service"], query["latitude"], query["longitude"], k=5)
        assert result["status"] != "unknown_service", f"Unknown service: {query['service']}"
        assert result["eligible_candidate_count"] > 0, f"No usable records: {query['service']}"
        nearby = result["recommendations"]
        assert all(item["distance_km"] <= result["policy"]["nearby_km"] for item in nearby)
        assert len({item["unique_id"] for item in nearby}) == len(nearby)
        assert [item["score"] for item in nearby] == sorted([item["score"] for item in nearby], reverse=True)
        for item in nearby + ([result["nearest_strong_option"]] if result["nearest_strong_option"] else []):
            parts = item["score_breakdown"]
            assert abs(item["score"] - parts["evidence_score"] * parts["location_weight"] * parts["distance_factor"]) < 1e-12
            assert item["quotes"]
        live_report.append({"query": query, "elapsed_seconds": perf_counter() - started, "result": result})
    print(json.dumps(live_report, indent=2, default=str, allow_nan=False))
else:
    print("Ten live queries and real-data result capture are PENDING.")
